# EXP-022 | Stacking v2 — LightGBM 메타 러너

EXP-016(LogisticRegression 메타 러너)을 LightGBM 메타 러너로 교체.
비선형 조합을 학습해 개별 모델의 강점을 더 잘 활용.

```
[LGB OOF]  ┐
[CAT OOF]  ├─→ LightGBM 메타 러너 → 최종 예측
[XGB OOF]  ┘
```

| 항목 | 내용 |
|---|---|
| Base 모델 | LGB(EXP-019) / CAT(EXP-011) / XGB(EXP-012) |
| Meta 러너 | LightGBM (소형, 과적합 방지) |
| 기준선 | EXP-016 Stacking-LR / EXP-020 앙상블-v4 |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score, precision_score, accuracy_score
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
SEED     = 42
N_FOLDS  = 5
EXP_NO   = 22
AUTHOR   = '조여진'
CV_STR   = f'Stratified {N_FOLDS}-Fold'

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')

train: (256351, 69)  /  test: (90067, 68)


## 1. 피처 준비

In [2]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_train, X_test = preprocess(train_fe, test_fe)
X_train = add_derived_features(X_train)
X_test  = add_derived_features(X_test)
y_train = train[TARGET]
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f'X_train: {X_train.shape}  /  X_test: {X_test.shape}')

X_train: (256351, 85)  /  X_test: (90067, 85)


## 2. Level-1: Base 모델 OOF 생성

In [3]:
LGB_PARAMS = dict(
    objective='binary', metric='auc', verbosity=-1, seed=SEED,
    num_threads=-1, is_unbalance=True, bagging_freq=1,
    learning_rate=0.035425966303284824, num_leaves=266, max_depth=5,
    min_child_samples=166, feature_fraction=0.5346439126449233,
    bagging_fraction=0.7122309235479091, lambda_l1=9.901034988600228,
    lambda_l2=2.213951873239442, min_gain_to_split=0.11418176854933762,
)
CAT_PARAMS = dict(
    iterations=2000, loss_function='Logloss', eval_metric='AUC',
    auto_class_weights='Balanced', random_seed=SEED,
    thread_count=-1, verbose=False, early_stopping_rounds=100,
    learning_rate=0.018758723768855998, depth=6,
    l2_leaf_reg=9.189608434163782, min_data_in_leaf=19,
    subsample=0.8170921295501483, colsample_bylevel=0.6936810336930781,
)
XGB_PARAMS = dict(
    n_estimators=2000, scale_pos_weight=neg_pos_ratio,
    eval_metric='auc', tree_method='hist', random_state=SEED,
    n_jobs=-1, verbosity=0, early_stopping_rounds=100,
    learning_rate=0.05520069867907647, max_depth=4, min_child_weight=59,
    subsample=0.7663066457187595, colsample_bytree=0.6581836436885355,
    reg_alpha=8.692038126211928, reg_lambda=0.23932562420374562,
)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_lgb  = np.zeros(len(X_train)); test_lgb = np.zeros(len(X_test))
oof_cat  = np.zeros(len(X_train)); test_cat = np.zeros(len(X_test))
oof_xgb  = np.zeros(len(X_train)); test_xgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    ds_tr = lgb.Dataset(X_tr, label=y_tr)
    ds_val = lgb.Dataset(X_val, label=y_val, reference=ds_tr)
    m = lgb.train(LGB_PARAMS, ds_tr, num_boost_round=2000, valid_sets=[ds_val],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[val_idx] = m.predict(X_val); test_lgb += m.predict(X_test) / N_FOLDS
    m = CatBoostClassifier(**CAT_PARAMS)
    m.fit(X_tr, y_tr, eval_set=Pool(X_val, y_val), use_best_model=True)
    oof_cat[val_idx] = m.predict_proba(X_val)[:, 1]; test_cat += m.predict_proba(X_test)[:, 1] / N_FOLDS
    m = xgb.XGBClassifier(**XGB_PARAMS)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = m.predict_proba(X_val)[:, 1]; test_xgb += m.predict_proba(X_test)[:, 1] / N_FOLDS

auc_lgb = roc_auc_score(y_train, oof_lgb)
auc_cat = roc_auc_score(y_train, oof_cat)
auc_xgb = roc_auc_score(y_train, oof_xgb)
print(f'Base OOF  LGB={auc_lgb:.5f}  CAT={auc_cat:.5f}  XGB={auc_xgb:.5f}')

Base OOF  LGB=0.74047  CAT=0.74015  XGB=0.74051


## 3. Level-2: LightGBM 메타 러너

3개 열(OOF) → LGB 메타 러너. 과적합 방지를 위해 `num_leaves=8` 소형으로 설정.

In [4]:
meta_train = np.stack([oof_lgb, oof_cat, oof_xgb], axis=1)
meta_test  = np.stack([test_lgb, test_cat, test_xgb], axis=1)

# 메타 러너 파라미터 — 피처가 3개뿐이므로 작은 모델 필요
META_PARAMS = dict(
    objective='binary', metric='auc', verbosity=-1, seed=SEED,
    num_threads=-1, is_unbalance=True,
    learning_rate=0.05,
    num_leaves=8,          # 피처 3개에 맞게 최소화
    min_child_samples=100,
    feature_fraction=1.0,  # 피처가 3개라 전부 사용
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l1=1.0,
    lambda_l2=1.0,
)

# 5-Fold OOF로 메타 러너 평가
meta_oof  = np.zeros(len(meta_train))
meta_test_pred = np.zeros(len(meta_test))

for tr_idx, val_idx in skf.split(meta_train, y_train):
    ds_tr  = lgb.Dataset(meta_train[tr_idx], label=y_train.iloc[tr_idx])
    ds_val = lgb.Dataset(meta_train[val_idx], label=y_train.iloc[val_idx],
                         reference=ds_tr)
    m = lgb.train(META_PARAMS, ds_tr, num_boost_round=500,
                  valid_sets=[ds_val],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(-1)])
    meta_oof[val_idx]   = m.predict(meta_train[val_idx])
    meta_test_pred     += m.predict(meta_test) / N_FOLDS

stacking_auc   = roc_auc_score(y_train, meta_oof)
stacking_prauc = average_precision_score(y_train, meta_oof)
stacking_f1    = f1_score(y_train, (meta_oof >= 0.5).astype(int))

BASELINE = 0.74046
print(f'Stacking(LGB meta) OOF AUC : {stacking_auc:.5f}')
print(f'Stacking(LGB meta) PR-AUC  : {stacking_prauc:.5f}')
print(f'Stacking(LGB meta) F1      : {stacking_f1:.5f}')
print(f'EXP-020 대비               : {stacking_auc - BASELINE:+.5f}')
print(f'EXP-016(LR) 대비           : (EXP-016 결과 확인 후 비교)')

Stacking(LGB meta) OOF AUC : 0.73999
Stacking(LGB meta) PR-AUC  : 0.44836
Stacking(LGB meta) F1      : 0.51733
EXP-020 대비               : -0.00047
EXP-016(LR) 대비           : (EXP-016 결과 확인 후 비교)


## 4. Submission 저장 & 실험 기록

In [5]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
sub['probability'] = meta_test_pred
auc_str   = f'{stacking_auc:.4f}'.replace('.', '')
out_fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{auc_str}.csv'
sub.to_csv(OUT_DIR / out_fname, index=False)
print(f'저장: {OUT_DIR / out_fname}')

저장: ../data/submissions/submission_exp022_조여진_07400.csv


In [6]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

oof_binary = (meta_oof >= 0.5).astype(int)
params_str = 'Base:LGB(EXP-019)+CAT(EXP-011)+XGB(EXP-012) | Meta:LGB(num_leaves=8)'
NOTES    = 'Stacking v2: 메타 러너 LogisticRegression → LightGBM 교체'
INSIGHTS = f'EXP-020 대비 {stacking_auc - BASELINE:+.5f} | base LGB={auc_lgb:.5f} CAT={auc_cat:.5f} XGB={auc_xgb:.5f}'

log_to_leaderboard(
    EXP_NO, AUTHOR, 'Stacking-v2(LGB+CAT+XGB→LGB)', params_str,
    f1_score(y_train, oof_binary), recall_score(y_train, oof_binary),
    precision_score(y_train, oof_binary), accuracy_score(y_train, oof_binary),
    stacking_auc, CV_STR, 'v1', X_train.shape[1],
    'is_unbalance / Balanced / scale_pos_weight',
    'N', None, 'notebooks/18_stacking_lgb_yjcho.ipynb', NOTES, INSIGHTS
)

[leaderboard.xlsx] EXP-022 기록 완료 (row 20)
